# Extract Volume Loss Metadata

This notebook processes the corrosion experiment results to extract volume loss values and corrosion modulator types. The original data is in Excel format with complex naming and structure. This notebook:

1. Loads volume loss data from Excel
2. Cleans and normalizes sample IDs (handles malformed entries)
3. Reshapes from wide to long format (one measurement per row)
4. Exports to machine-readable CSV format for downstream processing

In [ ]:
"""
Import required libraries
"""
from pathlib import Path
import pandas as pd

In [ ]:
"""
Define paths to source (Excel) and target (CSV) files
"""
# Project base path - adapt as needed for your installation
PROJECT_PATH = Path("/data/dust/user/dammal/beegfs.migration/repos/ai2")

# Source: Excel file with volume loss results from experiments
SOURCE_PATH = PROJECT_PATH / "rawdata/results/all-results.xlsx"

# Target: CSV file with processed volume loss data (for reference during training)
TARGET_PATH = PROJECT_PATH / "data/volume-loss/volume-loss.csv"

In [ ]:
"""
Load volume loss data from Excel
Columns: ID, modulator Name, and 4 volume loss measurements (one per image)
"""
data = pd.read_excel(
    SOURCE_PATH,
    skiprows=1,  # Skip title row
    usecols="A,C:G",  # Extract relevant columns
    names=["id", "name", "volumeloss1", "volumeloss2", "volumeloss3", "volumeloss4"]
)

print(f"Loaded {len(data)} samples")
print("\nFirst few entries:")
print(data.head())

,id,name,volumeloss1,volumeloss2,volumeloss3,volumeloss4
0,1-1-1,Tricarballylic acid,0.347,0.396,0.353,0.356
1,1-1-2,glycine,0.721,0.728,0.916,0.751
2,1-1-3,Tartronic acid,0.263,0.307,0.249,0.213
3,1-1-4,5-Nitrouracil,0.234,0.213,0.191,0.155
4,1-1-5,"2,5-Dimethyl-1,3,4-thiadiazole",0.262,0.348,0.330,0.296
...,...,...,...,...,...,...
294,#4-11-8,"2,4-Dimethoxybenzylakohol redo",0.932,0.549,0.276,0.215
295,#4-11-9,8-Anilino-1-naphthalenesulfonic acid redo,0.716,0.392,0.254,0.128
296,#4-11-10,"4-Hydroxy-3,5-dimethylbenzoic acid",0.682,0.553,0.495,0.367
297,#4-11-11,"3,5-Dimethoxybenzylalcohol",0.323,0.318,0.433,0.454


In [ ]:
"""
Check for malformed IDs in the data
Some samples have extra annotations like " (#3-1-1)" that need to be removed
"""
malformed_ids = data[data.id.str.contains("#", na=False)]
print(f"Found {len(malformed_ids)} entries with malformed IDs:")
print(malformed_ids)

,id,name,volumeloss1,volumeloss2,volumeloss3,volumeloss4
278,5-1-1 (#3-1-1),BIS-TRIS,0.572,0.596,0.510,0.599
284,#3-3-1,"2,4-Dimethoxybenzyl alcohol",0.585,0.498,0.556,0.364
285,#3-3-2,8-Anilino-1-naphthalenesulfonic acid,0.396,0.406,0.335,0.345
286,#3-3-3,"4-Hydroxy-3,5-dimethylbenzoic acid",0.446,0.427,0.337,0.363
287,#3-3-4,NaCl,0.496,0.450,0.446,0.386
288,#4-4-7,"3,5-Dimethoxyphenylacetic acid",0.439,0.487,0.454,0.529
289,#4-4-8,DL-4-Hydroxy-3-methoxymandelic acid,0.480,0.441,0.584,0.582
290,#4-4-9,6-Aminoindole,0.371,0.379,0.300,0.328
291,#4-4-10,5-Aminoindole,0.385,0.451,0.469,0.408
292,#4-4-11,NaCl,0.595,0.492,0.542,0.452


In [ ]:
"""
Clean ID formatting: remove unwanted extensions like " (#3-1-1)"
"""
data["id"] = data["id"].str.replace(" (#3-1-1)", "", regex=False)

In [ ]:
"""
Replace malformed markers: "#" should be "1" (e.g., "#-1-1" → "1-1-1")
This appears to be OCR or transcription errors in the original data
"""
data["id"] = data["id"].str.replace("#", "1", regex=False)

In [7]:
data

,id,name,volumeloss1,volumeloss2,volumeloss3,volumeloss4
0,1-1-1,Tricarballylic acid,0.347,0.396,0.353,0.356
1,1-1-2,glycine,0.721,0.728,0.916,0.751
2,1-1-3,Tartronic acid,0.263,0.307,0.249,0.213
3,1-1-4,5-Nitrouracil,0.234,0.213,0.191,0.155
4,1-1-5,"2,5-Dimethyl-1,3,4-thiadiazole",0.262,0.348,0.330,0.296
...,...,...,...,...,...,...
294,14-11-8,"2,4-Dimethoxybenzylakohol redo",0.932,0.549,0.276,0.215
295,14-11-9,8-Anilino-1-naphthalenesulfonic acid redo,0.716,0.392,0.254,0.128
296,14-11-10,"4-Hydroxy-3,5-dimethylbenzoic acid",0.682,0.553,0.495,0.367
297,14-11-11,"3,5-Dimethoxybenzylalcohol",0.323,0.318,0.433,0.454


In [ ]:
"""
Reshape data from wide to long format

Original format: one row per sample, 4 volume loss columns (for 4 measurements)
New format: one row per measurement, with image index appended to ID

This enables easy lookup of volume loss and modulator type by image ID during training
"""
data = data.melt(
    id_vars=["id", "name"],
    var_name="lossimage",
    value_name="volumeloss"
)

# Append image number (1-4) to sample ID
data["id"] = data["id"] + "-" + data["lossimage"].str[-1]

# Remove the intermediate column
data.drop(["lossimage"], axis=1, inplace=True)

print(f"Reshaped to {len(data)} measurement entries")

,id,name,volumeloss
0,1-1-1-1,Tricarballylic acid,0.347
1,1-1-2-1,glycine,0.721
2,1-1-3-1,Tartronic acid,0.263
3,1-1-4-1,5-Nitrouracil,0.234
4,1-1-5-1,"2,5-Dimethyl-1,3,4-thiadiazole",0.262
...,...,...,...
1191,14-11-8-4,"2,4-Dimethoxybenzylakohol redo",0.215
1192,14-11-9-4,8-Anilino-1-naphthalenesulfonic acid redo,0.128
1193,14-11-10-4,"4-Hydroxy-3,5-dimethylbenzoic acid",0.367
1194,14-11-11-4,"3,5-Dimethoxybenzylalcohol",0.454


In [ ]:
"""
Clean modulator names: remove line breaks that may have been introduced during data entry
"""
data["name"] = data["name"].str.replace("\n", "-", regex=False)

In [ ]:
"""
Export processed volume loss data to CSV
Format: id, name (modulator type), volumeloss
"""
data.to_csv(TARGET_PATH, index=False)
print(f"✓ Volume loss data saved to {TARGET_PATH}")
print(f"  Total entries: {len(data)}")
print(f"  Unique modulators: {data['name'].nunique()}")
print(f"\nFirst few entries:")
print(data.head())